ollama-python SDK
Repo: https://github.com/ollama/ollama-python/tree/main
Examples: https://github.com/ollama/ollama-python/tree/main/examples

In [ ]:
import asyncio
import ollama
from ollama import chat, generate
from pprint import pprint
from pydantic import BaseModel
from ollama import ChatResponse, GenerateResponse, EmbedResponse, Client, AsyncClient

## Single prompt

In [6]:
response: GenerateResponse = generate(model='llama3.1', prompt="Hello my name is Davide")
print(response.response)

Ciao Davide! (That's Italian for "hello") It's nice to meet you. Is there something I can help you with or would you like to chat?


## ChatResponse Attributes

In [ ]:
pprint([x for x in dir(response) if ((not x.startswith("_")) and (not callable(x)))]) # response object

## Conversation Chat

In [ ]:
rhyme_word = "dish"

messages = [
    {
        "role": "system",
        "content": "You are a expert rapper. You can find rhymes with a lot of words and construct rap lyrics. Keep your answers limited to a single sentence"
    },
    {
        "role": "user",
        "content": f"What rhymes with the word '{rhyme_word}'"
    },
]

response: ChatResponse = chat(model='llama3.1', messages=messages)
print("#1: ", response['message']['content'])

messages.append(response.message)
messages.append({"role": "user", "content": "Now make a rhyme with the word single and the previous word rhyme word."})
response: ChatResponse = chat(model='llama3.1', messages=messages)
print("#2: ", response['message']['content'])


## Streaming Response

In [ ]:

stream = chat(
    model='llama3.1',
    messages=[{'role': 'user', 'content': 'Why is the sky blue?'}],
    stream=True,
)

for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)

## Custom Client

In [ ]:
client = Client(
  host='http://localhost:11434',
  headers={'x-some-header': 'some-value'}
)
response = client.chat(model='llama3.1', messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
])

## Async Client

In [ ]:

async def async_chat():
  message = {'role': 'user', 'content': 'Why is the sky blue?'}
  response = await AsyncClient().chat(model='llama3.1', messages=[message])


await async_chat() # for notebook use this
# asyncio.run(async_chat()) # for python mode use this

async def async_chat_stream():
  message = {'role': 'user', 'content': 'Why is the sky blue?'}
  async for part in await AsyncClient().chat(model='llama3.1', messages=[message], stream=True):
    print(part['message']['content'], end='', flush=True)

await async_chat_stream() # for notebook use this
# asyncio.run(async_chat_stream()) # for python mode use this

# Tool Use

## Single Tool Call

In [34]:

def add_two_numbers(a: int, b: int) -> int:
  """
  Add two numbers

  Args:
    a (int): The first number
    b (int): The second number

  Returns:
    int: The sum of the two numbers
  """

  # The cast is necessary as returned tool call arguments don't always conform exactly to schema
  # E.g. this would prevent "what is 30 + 12" to produce '3012' instead of 42
  return int(a) + int(b)


def subtract_two_numbers(a: int, b: int) -> int:
  """
  Subtract two numbers
  """

  # The cast is necessary as returned tool call arguments don't always conform exactly to schema
  return int(a) - int(b)


# Tools can still be manually defined and passed into chat
subtract_two_numbers_tool = {
  'type': 'function',
  'function': {
    'name': 'subtract_two_numbers',
    'description': 'Subtract two numbers',
    'parameters': {
      'type': 'object',
      'required': ['a', 'b'],
      'properties': {
        'a': {'type': 'integer', 'description': 'The first number'},
        'b': {'type': 'integer', 'description': 'The second number'},
      },
    },
  },
}

messages = [{'role': 'user', 'content': 'What is three plus one?'}]
print('Prompt:', messages[0]['content'])

available_functions = {
  'add_two_numbers': add_two_numbers,
  'subtract_two_numbers': subtract_two_numbers,
}

response: ChatResponse = chat(
  'llama3.1',
  messages=messages,
  tools=[add_two_numbers, subtract_two_numbers_tool],
)

if response.message.tool_calls:
  # There may be multiple tool calls in the response
  for tool in response.message.tool_calls:
    # Ensure the function is available, and then call it
    if function_to_call := available_functions.get(tool.function.name):
      print('Calling function:', tool.function.name)
      print('Arguments:', tool.function.arguments)
      output = function_to_call(**tool.function.arguments)
      print('Function output:', output)
    else:
      print('Function', tool.function.name, 'not found')

# Only needed to chat with the model using the tool call results
if response.message.tool_calls:
  # Add the function response to messages for the model to use
  messages.append(response.message)
  messages.append({'role': 'tool', 'content': str(output), 'tool_name': tool.function.name})

  # Get final response from model with function outputs
  final_response = chat('llama3.1', messages=messages)
  print('Final response:', final_response.message.content)

else:
  print('No tool calls returned from model')

Prompt: What is three plus one?
Calling function: add_two_numbers
Arguments: {'a': 3, 'b': 1}
Function output: 4
Final response: The answer to 3 + 1 is 4.


## Multi-tool Call

In [ ]:
import random
from typing import Iterator


def get_temperature(city: str) -> int:
  """
  Get the temperature for a city in Celsius

  Args:
    city (str): The name of the city

  Returns:
    int: The current temperature in Celsius
  """
  # This is a mock implementation - would need to use a real weather API
  import random

  if city not in ['London', 'Paris', 'New York', 'Tokyo', 'Sydney']:
    return 'Unknown city'

  return str(random.randint(0, 35)) + ' degrees Celsius'


def get_conditions(city: str) -> str:
  """
  Get the weather conditions for a city
  """
  if city not in ['London', 'Paris', 'New York', 'Tokyo', 'Sydney']:
    return 'Unknown city'
  # This is a mock implementation - would need to use a real weather API
  conditions = ['sunny', 'cloudy', 'rainy', 'snowy']
  return random.choice(conditions)


available_functions = {
  'get_temperature': get_temperature,
  'get_conditions': get_conditions,
}


cities = ['London', 'Paris', 'New York', 'Tokyo', 'Sydney']
city = random.choice(cities)
city2 = random.choice(cities)
messages = [{'role': 'user', 'content': f'What is the temperature in {city}? and what are the weather conditions in {city2}?'}]
print('----- Prompt:', messages[0]['content'], '\n')

model = 'qwen3'
client = Client()
response: Iterator[ChatResponse] = client.chat(model, stream=True, messages=messages, tools=[get_temperature, get_conditions], think=True)

for chunk in response:
  if chunk.message.thinking:
    print(chunk.message.thinking, end='', flush=True)
  if chunk.message.content:
    print(chunk.message.content, end='', flush=True)
  if chunk.message.tool_calls:
    for tool in chunk.message.tool_calls:
      if function_to_call := available_functions.get(tool.function.name):
        print('\nCalling function:', tool.function.name, 'with arguments:', tool.function.arguments)
        output = function_to_call(**tool.function.arguments)
        print('> Function output:', output, '\n')

        # Add the assistant message and tool call result to the messages
        messages.append(chunk.message)
        messages.append({'role': 'tool', 'content': str(output), 'tool_name': tool.function.name})
      else:
        print('Function', tool.function.name, 'not found')

print('----- Sending result back to model \n')
if any(msg.get('role') == 'tool' for msg in messages):
  res = client.chat(model, stream=True, tools=[get_temperature, get_conditions], messages=messages, think=True)
  done_thinking = False
  for chunk in res:
    if chunk.message.thinking:
      print(chunk.message.thinking, end='', flush=True)
    if chunk.message.content:
      if not done_thinking:
        print('\n----- Final result:')
        done_thinking = True
      print(chunk.message.content, end='', flush=True)
    if chunk.message.tool_calls:
      # Model should be explaining the tool calls and the results in this output
      print('Model returned tool calls:')
      print(chunk.message.tool_calls)
else:
  print('No tool calls returned')

## Structured Output

In [ ]:

# Define the schema for the response
class FriendInfo(BaseModel):
  name: str
  age: int
  is_available: bool


class FriendList(BaseModel):
  friends: list[FriendInfo]


schema = {'type': 'object', 'properties': {'friends': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'age': {'type': 'integer'}, 'is_available': {'type': 'boolean'}}, 'required': ['name', 'age', 'is_available']}}}, 'required': ['friends']}
response = chat(
  model='llama3.1:8b',
  messages=[{'role': 'user', 'content': 'I have two friends. The first is Ollama 22 years old busy saving the world, and the second is Alonso 23 years old and wants to hang out. Return a list of friends in JSON format'}],
  format=FriendList.model_json_schema(),  # Use Pydantic to generate the schema or format=schema
  options={'temperature': 0},  # Make responses more deterministic
)

# Use Pydantic to validate the response
friends_response = FriendList.model_validate_json(response.message.content)
print(friends_response)

## Thinking / Reasoning

In [33]:
messages = [
  {
    'role': 'user',
    'content': 'What is 10 + 23 * 2?',
  },
]

response_think = chat("qwen3:1.7b", messages=messages, think=True)
response_nothink = chat("qwen3:1.7b", messages=messages, think=False)

print('Thinking:\n========\n\n' + response_think.message.thinking)
print('\nResponse:\n========\n\n' + response_think.message.content)
print('\n No Think Response:\n========\n\n' + response_nothink.message.content)

Thinking:

Okay, let's see. I need to figure out what 10 plus 23 multiplied by 2 is. Hmm, I remember from math class that there's something about the order of operations. Let me think. So, the problem is 10 + 23 * 2. 

Wait, right, the order of operations is parentheses, exponents, multiplication and division, and then addition and subtraction. So, I should do the multiplication before the addition, right? Because multiplication comes before addition in the order. So, first, let's handle the multiplication part. 

23 multiplied by 2. Let me calculate that. 23 times 2 is 46. So, now the equation becomes 10 + 46. Then, adding those together. 10 plus 46... that should be 56. 

But wait, let me double-check. Maybe I should write it down step by step to make sure I didn't make a mistake. So original problem: 10 + 23 * 2. According to order of operations, multiplication is done first. So 23 * 2 is indeed 46. Then add 10 to 46. 10 + 46 is 56. 

Hmm, seems straightforward. But maybe I should c

# Multi-modal

## Image

In [29]:
with open('player.png', 'rb') as file:
  response = ollama.chat(
    model='llama3.2-vision',
    messages=[
      {
        'role': 'user',
        'content': 'What color is the tie the man is wearing?',
        'images': [file.read()],
      },
    ],
  )
print(response['message']['content'])

The man in the image is wearing a dark blue tie.


## Embed

In [ ]:
output: EmbedResponse = ollama.embed(model='llama3.1', input='The sky is blue because of rayleigh scattering')
output_embeddings = output.embeddings

print(output)
print("Embedding dimension:", len(output.embeddings[0]))


batch_output: EmbedResponse = ollama.embed(model='llama3.1', input=['The sky is blue because of rayleigh scattering', 'Grass is green because of chlorophyll'])
print(batch_output)
print("Number of embeddings:", len(batch_output.embeddings))

model='llama3.1' created_at=None done=None done_reason=None total_duration=81895391 load_duration=28374244 prompt_eval_count=9 prompt_eval_duration=None eval_count=None eval_duration=None embeddings=[[-0.016079675, 0.014257161, 0.026730182, 0.009130004, -0.008106025, 0.01167082, -0.0096707065, -0.01326192, 0.0050849523, 0.014117213, 0.013962345, -0.016347263, 0.014427895, 0.03109242, 0.010318483, 0.02619438, 0.003142463, 0.01728637, -0.021327415, -0.011783264, -0.029904148, 0.006737447, -0.015632337, 0.0022309006, -0.009235364, 0.0059985765, 0.035871185, 0.013650114, 0.005001768, 0.0036046111, -0.017572653, -0.007826637, 0.0228011, -0.0031676623, 0.046079278, -0.0001130008, -0.0014948333, -0.014347356, -0.00019130636, -0.010011953, -0.0029663753, -0.010539756, 0.0024633396, -0.010841702, 0.019556927, 0.008405555, 0.002773786, -0.001103662, -0.0075227935, -0.026851228, -0.011387459, 0.008778119, -0.014848858, -0.01787863, 0.012673836, -0.0012644262, -0.01695321, -0.016309915, -0.0114480

## API

In [ ]:
print(ollama.list(), "\n####\n")
print(ollama.show("llama3.1"), "\n####\n")
print(ollama.ps(), "\n####\n")


# ollama.create(model='example', modelfile=modelfile) (using ModelFile)
# ollama.create(model='example', from_='gemma3', system="You are Mario from Super Mario Bros.")
# ollama.copy('gemma3', 'user/gemma3')
# ollama.delete('gemma3')
# ollama.pull('gemma3')
# ollama.push('user/gemma3')
#
#

models=[Model(model='llama3.2:1b', modified_at=datetime.datetime(2025, 7, 28, 12, 10, 22, 118636, tzinfo=TzInfo(+02:00)), digest='baf6a787fdffd633537aa2eb51cfd54cb93ff08e28040095462bb63daf552878', size=1321098329, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='1.2B', quantization_level='Q8_0')), Model(model='qwen3:4b', modified_at=datetime.datetime(2025, 6, 15, 9, 59, 54, 293770, tzinfo=TzInfo(+02:00)), digest='2bfd38a7daaf4b1037efe517ccb73d1a3bbd4822cf89f1a82be1569050a114e0', size=2620788260, details=ModelDetails(parent_model='', format='gguf', family='qwen3', families=['qwen3'], parameter_size='4.0B', quantization_level='Q4_K_M')), Model(model='qwen3:1.7b', modified_at=datetime.datetime(2025, 6, 7, 19, 5, 6, 890796, tzinfo=TzInfo(+02:00)), digest='8f68893c685c3ddff2aa3fffce2aa60a30bb2da65ca488b61fff134a4d1730e7', size=1359293444, details=ModelDetails(parent_model='', format='gguf', family='qwen3', families=['qwen3'], parameter